In [12]:
import numpy as np
import matplotlib.pyplot as plt

"""
Process
Inputs
    electric field source function
    magnetic field source function
    electric boundary conditions
    magnetic boundary conditions
    physical size
    physical step sizes
    3D array of relative permittivity constants
    total time
    time steps
    minimum wavelength
Check for input errors
    parameters positive
    physical step sizes below threshold
    time step size below threshold
    Array of permittivity constants same dimensions as total size / step size
    
"""

# Define constants
eps_0 = 8.8541878128e-12
mu_0 = 1.256637062e-6
c0 = 1 / np.sqrt(eps_0 * mu_0)
imp0 = np.sqrt(mu_0 / eps_0)

def fdtd(
    lambda_min, # in meters
    x_max, # in meters
    y_max, # in meters
    z_max, # in meters
    dx, # in meters
    dy, # in meters
    dz, # in meters
    t_max, # in seconds
    dt, # in seconds
    x_magnetic_source,
    y_magnetic_source,
    z_magnetic_source,
    x_magnetic_boundary,
    y_magnetic_boundary,
    z_magnetic_boundary,
    x_electric_source,
    y_electric_source,
    z_electric_source,
    x_electric_boundary,
    y_electric_boundary,
    z_electric_boundary,
    eps_r = None, # relative permittivities
):
    error = False

    # Validate positive parameters first
    positive_params = {
        "lambda_min": lambda_min,
        "x_max": x_max,
        "y_max": y_max,
        "z_max": z_max,
        "dx": dx,
        "dy": dy,
        "dz": dz,
        "t_max": t_max,
        "dt": dt,
    }

    for name, value in positive_params.items():
        if not np.isfinite(value) or value <= 0:
            print(f"{name} must be finite and positive.")
            error = True

    if (error):
        return

    # Define restrictions on parameters
    max_pos_step = lambda_min / 20
    max_time_step = 1 / (
        c0 * np.sqrt(
            (1 / dx ** 2) +
            (1 / dy ** 2) +
            (1 / dz ** 2)
        )
    )

    # Check if inputs have errors
    if (dx > max_pos_step):
        print(f"Error: dx is too high. Maximum dx: {max_pos_step}.")
        error = True

    if (dy > max_pos_step):
        print(f"Error: dy is too high. Maximum dy: {max_pos_step}.")
        error = True

    if (dz > max_pos_step):
        print(f"Error: dz is too high. Maximum dz: {max_pos_step}.")
        error = True

    if (dt > max_time_step):
        print(f"Error: dt is too high. Maximum dt: is {max_time_step}.")
        error = True

    if (error):
        return

    # Define position steps
    x_steps = int(np.ceil(x_max / dx))
    y_steps = int(np.ceil(y_max / dy))
    z_steps = int(np.ceil(z_max / dz))

    if (eps_r is None):
        eps_r = np.ones(
            (x_steps, y_steps, z_steps),
            dtype = float
        )
    else:
        eps_r = np.asarray(eps_r, dtype=float)
        if (eps_r.shape != (x_steps, y_steps, z_steps)):
            print(f"eps_r must have shape ({x_steps}, {y_steps}, {z_steps}).")
            error = True
        elif (not np.all(np.isfinite(eps_r)) or np.any(eps_r <= 0)):
            print("eps_r must contain finite, positive values.")
            error = True

    if (error):
        return

    # Define time steps
    time_steps = int(np.ceil(t_max / dt))

    eps = eps_r * eps_0

    # Initialize electric and magnetic fields
    # E_i calculated at position i + (di / 2), j, k and time t
    Ex = np.zeros((x_steps, y_steps, z_steps))
    Ey = np.zeros((x_steps, y_steps, z_steps))
    Ez = np.zeros((x_steps, y_steps, z_steps))

    # H_i calculated at position i, j + (dj / 2), k + (dk / 2) and time t + (dt / 2)
    Hx = np.zeros((x_steps, y_steps, z_steps))
    Hy = np.zeros((x_steps, y_steps, z_steps))
    Hz = np.zeros((x_steps, y_steps, z_steps))

    for t in range(time_steps):
        # update magnetic field boundaries
        Hx = x_magnetic_boundary(Hx)
        Hy = y_magnetic_boundary(Hy)
        Hz = z_magnetic_boundary(Hz)

        # update magnetic field
        # assume mu = mu0
        Hx[:, :-1, :-1] = Hx[:, :-1, :-1] + dt / mu_0 * (
            (Ey[:, :-1, 1:] - Ey[:, :-1, :-1]) / dz - (Ez[:, 1:, :-1] - Ez[:, :-1, :-1]) / dy
        )

        Hy[:-1, :, :-1] = Hy[:-1, :, :-1] + dt / mu_0 * (
            (Ez[1:, :, :-1] - Ez[:-1, :, :-1]) / dx - (Ex[:-1, :, 1:] - Ex[:-1, :, :-1]) / dz
        )

        Hz[:-1, :-1, :] = Hz[:-1, :-1, :] + dt / mu_0 * (
            (Ex[:-1, 1:, :] - Ex[:-1, :-1, :]) / dy - (Ey[1:, :-1, :] - Ey[:-1, :-1, :]) / dx
        )
    
        # magnetic field source
        Hx = x_magnetic_source(Hx)
        Hy = y_magnetic_source(Hy)
        Hz = z_magnetic_source(Hz)

        # update electric field boundaries
        Ex = x_electric_boundary(Ex)
        Ey = y_electric_boundary(Ey)
        Ez = z_electric_boundary(Ez)

        # update electric field
        Ex[:, 1:, 1:] = Ex[:, 1:, 1:] + dt / eps[:, 1:, 1:] * (
            (Hz[:, 1:, 1:] - Hz[:, :-1, 1:]) / dy - (Hy[:, 1:, 1:] - Hy[:, 1:, :-1]) / dz
        )
        
        Ey[1:, :, 1:] = Ey[1:, :, 1:] + dt / eps[1:, :, 1:] * (
            (Hx[1:, :, 1:] - Hx[1:, :, :-1]) / dz - (Hz[1:, :, 1:] - Hz[:-1, :, 1:]) / dx
        )
        
        Ez[1:, 1:, :] = Ez[1:, 1:, :] + dt / eps[1:, 1:, :] * (
            (Hy[1:, 1:, :] - Hy[:-1, 1:, :]) / dx - (Hx[1:, 1:, :] - Hx[1:, :-1, :]) / dy
        )
    
        # electric field source
        Ex = x_electric_source(Ex)
        Ey = y_electric_source(Ey)
        Ez = z_electric_source(Ez)
